# 02 - Modeling

**Task.** Predict whether a diabetic inpatient is readmitted within 30 days (`target = 1`).
**Final model.** Tuned CatBoost, with the decision threshold chosen on a patient-grouped validation split.

This notebook builds the whole training pipeline: feature engineering, preprocessing, model comparison, hyperparameter tuning, and threshold selection. The test set is used **only once**, at the very end, to report final results. Because the positive class is only about 11% of the data, **Average Precision (AP)** is the main selection metric, and ROC-AUC, Precision, Recall and F1 are reported as secondary metrics. All splits are grouped by `patient_nbr`, so the same patient never appears in both the training and validation folds. All artifacts are saved to `outputs/` and are read by the evaluation notebook (`03_evaluation.ipynb`).


In [1]:
import pandas as pd
import numpy as np
train_df = pd.read_csv("../data/train_clean.csv")
test_df = pd.read_csv("../data/test_clean.csv")

patient_groups = pd.read_csv(
    "../data/train_patient_groups.csv"
)["patient_nbr"]
assert len(train_df) == len(patient_groups)
assert patient_groups.notna().all()
assert train_df.columns.equals(test_df.columns)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Patient groups:", patient_groups.shape)

train_df.head()

Train shape: (79541, 32)
Test shape : (19802, 32)
Patient groups: (79541,)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glyburide,pioglitazone,rosiglitazone,insulin,change,diabetesMed,target,diag_1_group,diag_2_group,diag_3_group
0,Caucasian,Female,[0-10),Unknown,Unknown,1,1,Unknown,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,0,Diabetes,Missing,Missing
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,...,No,No,No,Up,Ch,Yes,0,Endocrine_other,Diabetes,Endocrine_other
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,...,No,No,No,No,No,Yes,0,Pregnancy_Childbirth,Diabetes,Supplementary_V
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,...,No,No,No,Up,Ch,Yes,0,Infectious,Diabetes,Circulatory
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,...,No,No,No,Steady,Ch,Yes,0,Neoplasms,Neoplasms,Diabetes


#Advanced feature


In [2]:
def add_advanced_features(df):
    df = df.copy()
    # 1. Total number of previous healthcare visits
    df["total_previous_visits"] = (
        df["number_outpatient"]
        + df["number_emergency"]
        + df["number_inpatient"]
    )
    # 2. Whether the patient had any previous inpatient visits
    df["has_prior_inpatient"] = (
        df["number_inpatient"] > 0
    ).astype(int)
    # 3. Whether the patient had any previous emergency visits
    df["has_prior_emergency"] = (
        df["number_emergency"] > 0
    ).astype(int)
    # 4. Average number of medications per hospital day
    df["medications_per_day"] = (
        df["num_medications"]
        / df["time_in_hospital"].clip(lower=1)
    )
    # 5. Average number of lab procedures per hospital day
    df["labs_per_day"] = (
        df["num_lab_procedures"]
        / df["time_in_hospital"].clip(lower=1)
    )
    # 6. Average number of procedures per hospital day
    df["procedures_per_day"] = (
        df["num_procedures"]
        / df["time_in_hospital"].clip(lower=1)
    )
    # 7. Overall treatment intensity
    df["treatment_intensity"] = (
        df["num_lab_procedures"]
        + df["num_procedures"]
        + df["num_medications"]
    )
    return df

train_fe = add_advanced_features(train_df)
test_fe = add_advanced_features(test_df)

print("Train shape before feature engineering:", train_df.shape)
print("Train shape after feature engineering :", train_fe.shape)

print("Test shape before feature engineering:", test_df.shape)
print("Test shape after feature engineering :", test_fe.shape)

Train shape before feature engineering: (79541, 32)
Train shape after feature engineering : (79541, 39)
Test shape before feature engineering: (19802, 32)
Test shape after feature engineering : (19802, 39)


## Baseline Models

Before training the baseline models, the target variable is separated from
the predictors. Numerical features are standardized, while categorical
features are transformed using One-Hot Encoding.

The preprocessing pipeline is fitted only on the training data to prevent
data leakage.

In [3]:
# Separate features and target
X_train = train_fe.drop(columns=["target"])
y_train = train_fe["target"]

X_test = test_fe.drop(columns=["target"])
y_test = test_fe["target"]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (79541, 38)
X_test : (19802, 38)
y_train: (79541,)
y_test : (19802,)


In [4]:
categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_ids = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in categorical_ids:
    if col in X_train.columns and col not in categorical_features:
        categorical_features.append(col)

numerical_features = [
    col for col in X_train.select_dtypes(include="number").columns
    if col not in categorical_features
]

assert set(numerical_features + categorical_features) == set(X_train.columns)

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'total_previous_visits', 'has_prior_inpatient', 'has_prior_emergency', 'medications_per_day', 'labs_per_day', 'procedures_per_day', 'treatment_intensity']
Categorical features: ['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'change', 'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group']


In [5]:
from sklearn.model_selection import GroupShuffleSplit

groups = patient_groups.to_numpy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

fit_idx, val_idx = next(
    gss.split(X_train, y_train, groups=groups)
)

X_fit = X_train.iloc[fit_idx]
X_val = X_train.iloc[val_idx]

y_fit = y_train.iloc[fit_idx]
y_val = y_train.iloc[val_idx]

groups_fit = groups[fit_idx]
groups_val = groups[val_idx]

shared_patients = len(set(groups_fit) & set(groups_val))
assert shared_patients == 0

print("Fit:", X_fit.shape)
print("Validation:", X_val.shape)
print("Shared patients:", shared_patients)
print("Fit positive rate:", round(y_fit.mean(), 4))
print("Validation positive rate:", round(y_val.mean(), 4))

Fit: (63807, 38)
Validation: (15734, 38)
Shared patients: 0
Fit positive rate: 0.1143
Validation positive rate: 0.1145


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("scaler", RobustScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


### Baseline Model Training

Three baseline classification models are trained:

- Logistic Regression
- Decision Tree
- Random Forest

No class balancing or hyperparameter tuning is applied at this stage.
The baseline results will be used as a reference for evaluating more advanced approaches.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

import pandas as pd

**Logistic Regression**

In [8]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

**Decision Tree**

In [9]:
decision_tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=20,
    random_state=42
))
])

**Random Forest**

In [10]:
random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
))
])

## Advanced Models

Advanced gradient boosting models are evaluated to capture more complex
non-linear relationships in the data.


In [11]:
!pip install -q catboost

**CatBoost**

In [12]:
from catboost import CatBoostClassifier
catboost_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function="Logloss",
        random_seed=42,
        verbose=0,
        allow_writing_files=False
    ))
])

## Imbalance Handling

The target variable is highly imbalanced, with the positive class representing
only about 11% of the training data.



**Class Weight**

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logistic_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [14]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42,
        max_depth=8,
        min_samples_leaf=20,
    ))
])

In [15]:
from sklearn.ensemble import RandomForestClassifier
random_forest_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
      n_estimators=200,
      max_depth=12,
      class_weight="balanced",
      min_samples_leaf=10,
      random_state=42,
      n_jobs=-1
))
])

In [16]:
from catboost import CatBoostClassifier
catboost_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function="Logloss",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=0,
        allow_writing_files=False   
    ))
])

# Model Selection with Group-Aware Cross-Validation



In [17]:
from sklearn.base import clone
candidates = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "Random Forest": random_forest_pipeline,
    "CatBoost": catboost_pipeline,

    "Logistic Regression + Class Weight": logistic_balanced_pipeline,
    "Decision Tree + Class Weight": decision_tree_balanced_pipeline,
    "Random Forest + Class Weight": random_forest_balanced_pipeline,
    "CatBoost + Class Weight": catboost_balanced_pipeline,
}

In [18]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)
cv_rows = []
for name, pipeline in candidates.items():
    scores = cross_val_score(
        clone(pipeline),
        X_fit,
        y_fit,
        groups=groups_fit,
        cv=cv,
        scoring="average_precision",
        n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "Mean CV AP": scores.mean(),
        "Std CV AP": scores.std()
    })
cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values("Mean CV AP", ascending=False)
    .reset_index(drop=True)
)

random_baseline_ap = y_fit.mean()
print(f"Random baseline AP: {random_baseline_ap:.4f}")
cv_results["AP / Random Baseline"] = (
    cv_results["Mean CV AP"] / random_baseline_ap
)

display(cv_results.round(4))

Random baseline AP: 0.1143


,Model,Mean CV AP,Std CV AP,AP / Random Baseline
0,CatBoost,0.2270,0.0052,1.9855
1,Random Forest,0.2192,0.0036,1.9169
2,CatBoost + Class Weight,0.2161,0.0022,1.8902
3,Random Forest + Class Weight,0.2109,0.0008,1.8446
4,Logistic Regression,0.2091,0.0031,1.8284
5,Logistic Regression + Class Weight,0.2073,0.0029,1.8132
6,Decision Tree,0.1920,0.0012,1.6789
7,Decision Tree + Class Weight,0.1899,0.0020,1.6608


In [19]:
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

validation_rows = []

for name, pipeline in candidates.items():
    print("Running:", name, flush=True)

    model = clone(pipeline)
    model.fit(X_fit, y_fit)

    positive_col = list(model.classes_).index(1)
    val_proba = model.predict_proba(X_val)[:, positive_col]
    val_pred = (val_proba >= 0.5).astype(int)

    validation_rows.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_val, val_proba),
        "AP": average_precision_score(y_val, val_proba),
    })

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values("AP", ascending=False)
    .reset_index(drop=True)
)

display(validation_results.round(4))

Running: Logistic Regression


e:\study\diabetes-readmission-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Running: Decision Tree
Running: Random Forest
Running: CatBoost
Running: Logistic Regression + Class Weight


e:\study\diabetes-readmission-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Running: Decision Tree + Class Weight
Running: Random Forest + Class Weight
Running: CatBoost + Class Weight


,Model,ROC-AUC,AP
0,CatBoost,0.6779,0.2328
1,CatBoost + Class Weight,0.6690,0.2248
2,Random Forest,0.6753,0.2209
3,Logistic Regression,0.6653,0.2132
4,Decision Tree,0.6483,0.2122
5,Logistic Regression + Class Weight,0.6671,0.2120
6,Random Forest + Class Weight,0.6701,0.2071
7,Decision Tree + Class Weight,0.6437,0.2052


In [20]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

best_pipeline = candidates["CatBoost"]

param_dist = {
    "classifier__depth": randint(4, 10),
    "classifier__learning_rate": uniform(0.01, 0.2),
    "classifier__l2_leaf_reg": uniform(1, 10),
    "classifier__iterations": randint(200, 800),
}

cv_search = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=clone(best_pipeline),
    param_distributions=param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=cv_search,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(
    X_fit,
    y_fit,
    groups=groups_fit
)

print("Best params:", search.best_params_)
print("Best CV AP:", search.best_score_)

tuned_model = search.best_estimator_

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params: {'classifier__depth': 6, 'classifier__iterations': 405, 'classifier__l2_leaf_reg': np.float64(4.910606075732408), 'classifier__learning_rate': np.float64(0.04644721755761247)}
Best CV AP: 0.23420541477048928


In [21]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
best_params = {
    key.removeprefix("classifier__"): (
        value.item() if isinstance(value, np.generic) else value
    )
    for key, value in search.best_params_.items()
}

def make_final_pipeline():
    return Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", CatBoostClassifier(
            **best_params,
            loss_function="Logloss",
            random_seed=42,
            verbose=0
        ))
    ])

val_fit = make_final_pipeline()
val_fit.fit(X_fit, y_fit)

positive_col = list(val_fit.classes_).index(1)
val_proba = val_fit.predict_proba(X_val)[:, positive_col]

precision, recall, thresholds = precision_recall_curve(
    y_val, val_proba, pos_label=1
)

denom = precision[:-1] + recall[:-1]
f1_scores = np.divide(
    2 * precision[:-1] * recall[:-1],
    denom,
    out=np.zeros_like(denom),
    where=denom != 0
)

best_threshold = float(thresholds[np.argmax(f1_scores)])
print(f"Selected threshold: {best_threshold:.4f}")
final_model = make_final_pipeline()
final_model.fit(X_train, y_train)

positive_col = list(final_model.classes_).index(1)
test_proba = final_model.predict_proba(X_test)[:, positive_col]
y_pred_final = (test_proba >= best_threshold).astype(int)

print("\n=== FINAL TEST RESULTS ===")
print("Model: Tuned CatBoost")
print(f"Threshold: {best_threshold:.4f}")
print(f"Test AP: {average_precision_score(y_test, test_proba):.4f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_proba):.4f}")
print(classification_report(
    y_test,
    y_pred_final,
    digits=4,
    zero_division=0
))

print("Confusion matrix:\n", confusion_matrix(
    y_test, y_pred_final, labels=[0, 1]
))

Selected threshold: 0.1436

=== FINAL TEST RESULTS ===
Model: Tuned CatBoost
Threshold: 0.1436
Test AP: 0.2453
Test ROC-AUC: 0.6765
              precision    recall  f1-score   support

           0     0.9178    0.8022    0.8561     17586
           1     0.2149    0.4296    0.2865      2216

    accuracy                         0.7605     19802
   macro avg     0.5663    0.6159    0.5713     19802
weighted avg     0.8391    0.7605    0.7924     19802

Confusion matrix:
 [[14108  3478]
 [ 1264   952]]


In [23]:
import json
from pathlib import Path
import joblib

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(
    output_dir / "cv_results.csv",
    index=False
)
validation_results.to_csv(
    output_dir / "validation_results.csv",
    index=False
)
pd.DataFrame({
    "y_true": np.asarray(y_test),
    "probability": test_proba,
    "prediction": y_pred_final
}).to_csv(
    output_dir / "test_predictions.csv",
    index=False
)

test_metrics = pd.DataFrame([{
    "Model": "Tuned CatBoost",
    "Threshold": best_threshold,
    "AP": average_precision_score(y_test, test_proba),
    "ROC-AUC": roc_auc_score(y_test, test_proba),
    "Precision": precision_score(
        y_test, y_pred_final, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred_final, zero_division=0
    ),
    "F1": f1_score(
        y_test, y_pred_final, zero_division=0
    )
}])

test_metrics.to_csv(
    output_dir / "test_metrics.csv",
    index=False
)

with open(
    output_dir / "best_params.json",
    "w"
) as f:
    json.dump({
        "model": "CatBoost",
        "best_params": best_params,
        "best_cv_ap": float(search.best_score_),
        "threshold": float(best_threshold),
        "cv_folds": 3,
        "random_state": 42
    }, f, indent=4)

joblib.dump(
    final_model,
    output_dir / "final_model.joblib"
)

print("All outputs saved successfully!")
print("\nSaved files:")
for file in output_dir.iterdir():
    print("-", file.name)

All outputs saved successfully!

Saved files:
- best_params.json
- cv_results.csv
- final_model.joblib
- test_metrics.csv
- test_predictions.csv
- validation_results.csv
